In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## चरण 1: संरचित आउटपुटहरूको लागि Pydantic मोडेलहरू परिभाषित गर्नुहोस्

यी मोडेलहरूले **स्कीमा** परिभाषित गर्छ जुन एजेन्टहरूले फर्काउनेछन्। Pydantic सँग `response_format` प्रयोग गर्दा सुनिश्चित हुन्छ:
- ✅ प्रकार-सुरक्षित डेटा निकासी
- ✅ स्वचालित प्रमाणीकरण
- ✅ स्वतन्त्र-पाठ प्रतिक्रियाहरूबाट कुनै पनि पार्सिंग त्रुटि हुँदैन
- ✅ फिल्डहरूको आधारमा सजिलो सर्तीय रूटिङ


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## चरण २: होटल बुकिंग उपकरण सिर्जना गर्नुहोस्

यो उपकरण **availability_agent** ले कोठा उपलब्ध छन् कि छैनन् जाँच गर्न कल गर्नेछ। हामी `@ai_function` डेकोरेटर प्रयोग गर्छौं:
- पायथन फङ्सनलाई AI-कलयोग्य उपकरणमा परिवर्तन गर्न
- LLM का लागि स्वचालित JSON स्किमामा उत्पन्न गर्न
- प्यारामिटर परीक्षण गर्न ह्यान्डल गर्न
- एजेन्टहरूले स्वचालित रूपमा कल गर्न सक्षम बनाउन

यस डेमोका लागि:
- **स्टकहोम, सिएटल, टोक्यो, लन्डन, एम्स्टर्ड्याम** → कोठाहरू छन् ✅
- **अन्य सबै शहरहरू** → कोठाहरू छैनन् ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## चरण 3: रुटिङको लागि अवस्था कार्यहरू परिभाषित गर्नुहोस्

यी कार्यहरूले एजेन्टको प्रतिक्रिया निरीक्षण गर्छन् र कार्यप्रवाहमा कुन बाटो अपनाउने निर्णय गर्छन्।

**मुख्य ढाँचा:**
1. सन्देश `AgentExecutorResponse` हो कि होइन जाँच्नुहोस्
2. संरचित आउटपुट (Pydantic मोडेल) पार्स गर्नुहोस्
3. रुटिङ नियन्त्रण गर्न `True` वा `False` फर्काउनुहोस्

कार्यप्रवाहले यी अवस्थाहरू **एजहरूमा** मूल्यांकन गर्नेछ र निर्णय गर्नेछ कि अर्को कुन एक्जिक्युटरलाई बोलाउँछ।


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## चरण ४: अनुकूल प्रदर्शन कार्यान्वयनकर्ता सिर्जना गर्नुहोस्

कार्यान्वयनकर्ता कार्यप्रवाहका कम्पोनेंटहरू हुन् जसले रूपान्तरणहरू वा साइड इफेक्टहरू प्रदर्शन गर्छन्। हामी अन्तिम परिणाम देखाउन `@executor` डेकोरेटर प्रयोग गरी अनुकूल कार्यान्वयनकर्ता सिर्जना गर्छौं।

**प्रमुख अवधारणाहरू:**
- `@executor(id="...")` - कार्यप्रवाह कार्यान्वयनकर्ताका रूपमा फङ्क्शन दर्ता गर्छ
- `WorkflowContext[Never, str]` - इनपुट/आउटपुटका लागि प्रकार संकेतहरू
- `ctx.yield_output(...)` - अन्तिम कार्यप्रवाह परिणाम फर्काउँछ


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## चरण ५: वातावरण चरहरू लोड गर्नुहोस्

LLM क्लाइन्ट कन्फिगर गर्नुहोस्। यो उदाहरणले काम गर्छ:
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — सिफारिस गरिएको)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## चरण ६: संरचित आउटपुटहरूसहित AI एजेन्टहरू सिर्जना गर्नुहोस्

हामी **तीन विशेषज्ञ एजेन्टहरू** सिर्जना गर्छौं, प्रत्येकलाई `AgentExecutor` भित्र ओढाइएको:

१. **availability_agent** - उपकरण प्रयोग गरी होटेल उपलब्धता जाँच गर्छ
२. **alternative_agent** - विकल्प शहरहरू सुझाव दिन्छ (जब कोठा उपलब्ध छैन)
३. **booking_agent** - बुकिङ प्रोत्साहन गर्छ (जब कोठाहरू उपलब्ध छन्)

**मुख्य विशेषताहरू:**
- `tools=[hotel_booking]` - एजेन्टलाई उपकरण प्रदान गर्छ
- `response_format=PydanticModel` - संरचित JSON आउटपुटमा बाध्य पार्छ
- `AgentExecutor(..., id="...")` - कार्यप्रवाह प्रयोगका लागि एजेन्टलाई ओढाउँछ


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## चरण ७: सशर्त एजहरू सहित कार्यप्रवाह निर्माण गर्नुहोस्

अब हामी सशर्त मार्गनिर्देशनसहित ग्राफ निर्माण गर्न `WorkflowBuilder` प्रयोग गर्छौं:

**कार्यप्रवाह संरचना:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**मुख्य विधिहरू:**
- `.set_start_executor(...)` - प्रवेश बिन्दु सेट गर्छ
- `.add_edge(from, to, condition=...)` - सशर्त एज थप्छ
- `.build()` - कार्यप्रवाह पक्का गर्छ


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## चरण ८: टेस्ट केस १ चलाउनुहोस् - शहर बिना उपलब्धता (पेरिस)

हामीले पेरिसमा होटलहरू अनुरोध गरेर **कुनै उपलब्धता छैन** बाटो जाँचौ (हाम्रो सिमुलेशनमा त्यहाँ कोठाहरू उपलब्ध छैनन्)।


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## चरण ९: परीक्षण केस २ चलाउनुहोस् - उपलब्धता सहितको शहर (स्टकहोम)

अब हामी **उपलब्धता** बाटो जाँच गर्नेछौं स्टकहोममा होटलहरूको अनुरोध गरेर (जसमा हाम्रो सिमुलेशनमा कोठाहरू छन्)।


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## प्रमुख सिकाइहरू र आगामी कदमहरू

### ✅ तपाईले के सिक्नुभएको छ:

1. **WorkflowBuilder ढाँचा**
   - प्रवेश बिन्दु परिभाषित गर्न `.set_start_executor()` प्रयोग गर्नुहोस्
   - सर्त अनुसार रुटिङका लागि `.add_edge(from, to, condition=...)` प्रयोग गर्नुहोस्
   - workflow अन्तिम गर्न `.build()` कल गर्नुहोस्

2. **सर्त अनुसार रुटिङ**
   - Condition function हरू `AgentExecutorResponse` निरीक्षण गर्छन्
   - रुटिङ निर्णय गर्न संरचित आउटपुट पार्स गर्छन्
   - एज सक्रिय गर्न `True` फिर्ता दिन्छ, छोड्न `False`

3. **टूल एकीकरण**
   - Python function हरूलाई AI tool मा परिवर्तन गर्न `@ai_function` प्रयोग गर्नुहोस्
   - आवश्यक पर्दा एजेन्टहरूले टूलहरू स्वचालित रूपमा कल गर्छन्
   - टूलहरू JSON फिर्ता गर्छन् जसलाई एजेन्टहरू पार्स गर्न सक्छन्

4. **संरचित आउटपुटहरू**
   - प्रकार सुरक्षित डेटा निकाशका लागि Pydantic मोडेलहरू प्रयोग गर्नुहोस्
   - एजेन्टहरू सिर्जना गर्दा `response_format=MyModel` सेट गर्नुहोस्
   - प्रतिक्रिया पार्स गर्न `Model.model_validate_json()` प्रयोग गर्नुहोस्

5. **कस्टम एक्सिक्युटरहरू**
   - workflow कम्पोनेन्टहरू बनाउन `@executor(id="...")` प्रयोग गर्नुहोस्
   - एक्सिक्युटरहरूले डेटा रूपान्तरण वा साइड इफेक्ट गर्न सक्छन्
   - workflow परिणाम उत्पादन गर्न `ctx.yield_output()` प्रयोग गर्नुहोस्

### 🚀 वास्तविक संसारका अनुप्रयोगहरू:

- **यात्रा बुकिङ**: उपलब्धता जाँच, विकल्प सुझाव, तुलना गर्नुहोस्
- **ग्राहक सेवा**: समस्या प्रकार, भावना, प्राथमिकता अनुसार रुट गर्नुहोस्
- **इ-कॉमर्स**: सूची जाँच्नुहोस्, विकल्प सुझाव दिनुहोस्, आदेश प्रक्रिया गर्नुहोस्
- **सामग्री moderation**: विषाक्तता स्कोरहरू, प्रयोगकर्ता झण्डा अनुसार रुट गर्नुहोस्
- **अनुमोदन workflow**: रकम, प्रयोगकर्ता भूमिका, जोखिम स्तर अनुसार रुट गर्नुहोस्
- **बहु-चरण प्रक्रिया**: डेटा गुणस्तर, पूर्णता अनुसार रुट गर्नुहोस्

### 📚 आगामी कदमहरू:

- थप जटिल सर्तहरू थप्नुहोस् (बहु मापदण्डहरू)
- workflow स्थिति व्यवस्थापनका साथ लूपहरू कार्यान्वयन गर्नुहोस्
- पुन: प्रयोगयोग्य कम्पोनेन्टहरूका लागि उप-workflow थप्नुहोस्
- वास्तविक API हरूसँग एकीकृत गर्नुहोस् (होटल बुकिङ, सूची प्रणालीहरू)
- त्रुटि व्यवस्थापन र फेलब्याक मार्गहरू थप गर्नुहोस्
- बिल्ट-इन भिजुअलाइजेशन टूलहरूसँग workflow हरू दृश्यात्मक बनाउनुहोस्


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
